<a href="https://colab.research.google.com/github/sajid-ahsan-seyam-26/Air_Ticket_management_system/blob/main/foodpanda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load + basic overview

In [1]:
pip install pandas

In [5]:
from google.colab import files
uploaded = files.upload()

Saving Foodpanda Analysis Dataset.csv to Foodpanda Analysis Dataset.csv


In [9]:
import pandas as pd

df = pd.read_csv("Foodpanda Analysis Dataset.csv")

print(df.head(5))
print(df.tail(5))
print(df.shape)
print(df.columns)
print(df.info())
print(df.describe())

  customer_id  gender     age      city signup_date order_id order_date  \
0       C5663    Male   Adult  Peshawar   1/14/2024    O9663  8/23/2023   
1       C2831    Male   Adult    Multan    7/7/2024    O6831  8/23/2023   
2       C2851   Other  Senior    Multan   6/20/2025    O6851  8/23/2023   
3       C1694  Female  Senior  Peshawar    9/5/2023    O5694  8/23/2023   
4       C4339   Other  Senior    Lahore  12/29/2023    O8339  8/24/2023   

  restaurant_name dish_name category  quantity    price payment_method  \
0      McDonald's    Burger  Italian         5  1478.27           Cash   
1             KFC    Burger  Italian         3   956.04         Wallet   
2       Pizza Hut     Fries  Italian         2   882.51           Cash   
3          Subway     Pizza  Dessert         4   231.30           Card   
4             KFC  Sandwich  Dessert         1  1156.69           Cash   

   order_frequency last_order_date  loyalty_points   churned  rating  \
0               38       7/19/20

# Date column clean + total sales column

In [10]:
import pandas as pd

df = pd.read_csv("Foodpanda Analysis Dataset.csv")

# date columns convert
date_cols = ["signup_date", "order_date", "last_order_date", "rating_date"]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], format="%m/%d/%Y", errors="coerce")

# total order value
df["total_sales"] = df["quantity"] * df["price"]

print(df.head())

  customer_id  gender     age      city signup_date order_id order_date  \
0       C5663    Male   Adult  Peshawar  2024-01-14    O9663 2023-08-23   
1       C2831    Male   Adult    Multan  2024-07-07    O6831 2023-08-23   
2       C2851   Other  Senior    Multan  2025-06-20    O6851 2023-08-23   
3       C1694  Female  Senior  Peshawar  2023-09-05    O5694 2023-08-23   
4       C4339   Other  Senior    Lahore  2023-12-29    O8339 2023-08-24   

  restaurant_name dish_name category  ...    price  payment_method  \
0      McDonald's    Burger  Italian  ...  1478.27            Cash   
1             KFC    Burger  Italian  ...   956.04          Wallet   
2       Pizza Hut     Fries  Italian  ...   882.51            Cash   
3          Subway     Pizza  Dessert  ...   231.30            Card   
4             KFC  Sandwich  Dessert  ...  1156.69            Cash   

  order_frequency  last_order_date loyalty_points   churned rating  \
0              38       2025-07-19            238    Activ

# Missing value, duplicate, data type check

In [11]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nDuplicate order_id:")
print(df["order_id"].duplicated().sum())

print("\nData types:")
print(df.dtypes)

Missing values:
customer_id        0
gender             0
age                0
city               0
signup_date        0
order_id           0
order_date         0
restaurant_name    0
dish_name          0
category           0
quantity           0
price              0
payment_method     0
order_frequency    0
last_order_date    0
loyalty_points     0
churned            0
rating             0
rating_date        0
delivery_status    0
total_sales        0
dtype: int64

Duplicate rows:
0

Duplicate order_id:
0

Data types:
customer_id                object
gender                     object
age                        object
city                       object
signup_date        datetime64[ns]
order_id                   object
order_date         datetime64[ns]
restaurant_name            object
dish_name                  object
category                   object
quantity                    int64
price                     float64
payment_method             object
order_frequency             int64

# City-wise sales report

In [13]:
city_sales = df.groupby("city").agg(
    total_orders=("order_id", "count"),
    total_quantity=("quantity", "sum"),
    total_revenue=("total_sales", "sum"),
    avg_rating=("rating", "mean")
).reset_index()

city_sales = city_sales.sort_values("total_revenue", ascending=False)

print(city_sales)

        city  total_orders  total_quantity  total_revenue  avg_rating
3     Multan          1256            3782     3052313.80    3.004777
2     Lahore          1217            3686     2916372.35    2.989318
4   Peshawar          1195            3568     2906686.38    2.993305
0  Islamabad          1187            3551     2835310.27    2.994945
1    Karachi          1145            3363     2633229.28    3.001747
        city  total_orders  total_quantity  total_revenue  avg_rating
3     Multan          1256            3782     3052313.80    3.004777
2     Lahore          1217            3686     2916372.35    2.989318
4   Peshawar          1195            3568     2906686.38    2.993305
0  Islamabad          1187            3551     2835310.27    2.994945
1    Karachi          1145            3363     2633229.28    3.001747


# Category-wise sales report

In [14]:
category_sales = df.groupby("category").agg(
    total_orders=("order_id", "count"),
    total_revenue=("total_sales", "sum"),
    avg_price=("price", "mean"),
    avg_rating=("rating", "mean")
).reset_index()

category_sales = category_sales.sort_values("total_revenue", ascending=False)

print(category_sales)

      category  total_orders  total_revenue   avg_price  avg_rating
4      Italian          1236     3006032.50  813.529960    2.978964
1  Continental          1211     2928524.26  803.368382    2.997523
0      Chinese          1198     2889236.97  806.329324    3.006678
3    Fast Food          1222     2870929.69  793.483453    2.954992
2      Dessert          1133     2649188.66  784.755366    3.050309


# Delivery status analysis

In [15]:
delivery_report = df.groupby("delivery_status").agg(
    total_orders=("order_id", "count"),
    total_revenue=("total_sales", "sum"),
    avg_rating=("rating", "mean")
).reset_index()

print(delivery_report)

  delivery_status  total_orders  total_revenue  avg_rating
0       Cancelled          1968     4652141.25    2.965447
1         Delayed          1972     4672798.35    3.027890
2       Delivered          2060     5018972.48    2.997087


# Payment method analysis

In [16]:
payment_report = df.groupby("payment_method").agg(
    total_orders=("order_id", "count"),
    total_revenue=("total_sales", "sum")
).reset_index()

payment_report = payment_report.sort_values("total_revenue", ascending=False)

print(payment_report)

  payment_method  total_orders  total_revenue
1           Cash          2039     4960639.76
2         Wallet          1959     4726157.48
0           Card          2002     4657114.84


# Monthly sales trend

In [17]:
df["order_month"] = df["order_date"].dt.to_period("M").astype(str)

monthly_sales = df.groupby("order_month").agg(
    total_orders=("order_id", "count"),
    total_revenue=("total_sales", "sum")
).reset_index()

print(monthly_sales)

   order_month  total_orders  total_revenue
0      2023-08            77      173278.01
1      2023-09           274      696622.20
2      2023-10           262      665298.47
3      2023-11           226      531467.85
4      2023-12           260      671064.46
5      2024-01           260      672698.48
6      2024-02           225      502002.28
7      2024-03           236      558305.99
8      2024-04           252      634813.26
9      2024-05           232      556435.44
10     2024-06           247      616952.97
11     2024-07           265      617071.18
12     2024-08           263      562486.28
13     2024-09           274      600866.88
14     2024-10           246      590091.22
15     2024-11           255      625472.42
16     2024-12           239      591899.77
17     2025-01           255      643669.36
18     2025-02           240      594787.21
19     2025-03           279      683896.55
20     2025-04           235      515852.35
21     2025-05           231    

# Top 10 customers by revenue

In [18]:
top_customers = df.groupby("customer_id").agg(
    total_orders=("order_id", "count"),
    total_spent=("total_sales", "sum"),
    loyalty_points=("loyalty_points", "max"),
    avg_rating=("rating", "mean")
).reset_index()

top_customers = top_customers.sort_values("total_spent", ascending=False).head(10)

print(top_customers)

     customer_id  total_orders  total_spent  loyalty_points  avg_rating
2134       C3134             1      7496.85             312         5.0
1515       C2515             1      7495.05             352         5.0
5021       C6021             1      7494.65             304         3.0
1857       C2857             1      7493.60              70         2.0
2320       C3320             1      7491.80             103         5.0
4725       C5725             1      7486.50              63         5.0
4658       C5658             1      7477.30             179         5.0
5009       C6009             1      7473.40             407         5.0
4206       C5206             1      7471.75              61         4.0
5288       C6288             1      7468.20              38         3.0


# Active vs Inactive customer analysis

In [19]:
churn_report = df.groupby("churned").agg(
    total_customers=("customer_id", "nunique"),
    total_orders=("order_id", "count"),
    total_revenue=("total_sales", "sum"),
    avg_loyalty_points=("loyalty_points", "mean"),
    avg_rating=("rating", "mean")
).reset_index()

print(churn_report)

    churned  total_customers  total_orders  total_revenue  avg_loyalty_points  \
0    Active             3016          3016     7271237.07          248.228448   
1  Inactive             2984          2984     7072675.01          252.139745   

   avg_rating  
0    2.972812  
1    3.021113  


# Restaurant-wise performance

In [20]:
restaurant_report = df.groupby("restaurant_name").agg(
    total_orders=("order_id", "count"),
    total_revenue=("total_sales", "sum"),
    avg_rating=("rating", "mean")
).reset_index()

restaurant_report = restaurant_report.sort_values("total_revenue", ascending=False)

print(restaurant_report.head(10))

  restaurant_name  total_orders  total_revenue  avg_rating
3       Pizza Hut          1224     3002380.59    2.949346
4          Subway          1260     2998014.27    3.087302
1             KFC          1224     2954233.45    2.974673
0     Burger King          1151     2752780.87    2.966116
2      McDonald's          1141     2636502.90    3.002629


# Simple filtering examples

In [21]:
# only delayed orders
delayed_orders = df[df["delivery_status"] == "Delayed"]
print(delayed_orders.head())

# only cancelled orders
cancelled_orders = df[df["delivery_status"] == "Cancelled"]
print(cancelled_orders.head())

# high value orders
high_value_orders = df[df["total_sales"] > 2000]
print(high_value_orders.head())

# active customers only
active_customers = df[df["churned"] == "Active"]
print(active_customers.head())

  customer_id  gender     age      city signup_date order_id order_date  \
1       C2831    Male   Adult    Multan  2024-07-07    O6831 2023-08-23   
2       C2851   Other  Senior    Multan  2025-06-20    O6851 2023-08-23   
3       C1694  Female  Senior  Peshawar  2023-09-05    O5694 2023-08-23   
8       C5146   Other  Senior   Karachi  2024-12-17    O9146 2023-08-24   
9       C3579    Male   Adult   Karachi  2023-10-13    O7579 2023-08-24   

  restaurant_name dish_name     category  ...  payment_method  \
1             KFC    Burger      Italian  ...          Wallet   
2       Pizza Hut     Fries      Italian  ...            Cash   
3          Subway     Pizza      Dessert  ...            Card   
8             KFC     Pizza      Dessert  ...          Wallet   
9       Pizza Hut     Pizza  Continental  ...            Cash   

   order_frequency last_order_date  loyalty_points   churned  rating  \
1               24      2024-11-25              81    Active       2   
2             

# Rating analysis by city and category

In [22]:
rating_analysis = df.groupby(["city", "category"]).agg(
    avg_rating=("rating", "mean"),
    total_orders=("order_id", "count")
).reset_index()

print(rating_analysis.head(20))

         city     category  avg_rating  total_orders
0   Islamabad      Chinese    2.891566           249
1   Islamabad  Continental    2.932489           237
2   Islamabad      Dessert    3.068376           234
3   Islamabad    Fast Food    2.929515           227
4   Islamabad      Italian    3.154167           240
5     Karachi      Chinese    3.023697           211
6     Karachi  Continental    2.933054           239
7     Karachi      Dessert    3.156522           230
8     Karachi    Fast Food    3.020576           243
9     Karachi      Italian    2.873874           222
10     Lahore      Chinese    3.083333           264
11     Lahore  Continental    3.082251           231
12     Lahore      Dessert    2.971014           207
13     Lahore    Fast Food    2.923077           234
14     Lahore      Italian    2.893238           281
15     Multan      Chinese    2.991266           229
16     Multan  Continental    3.014815           270
17     Multan      Dessert    3.115741        

# Export report to Excel

In [23]:
with pd.ExcelWriter("foodpanda_analysis_report.xlsx") as writer:
    city_sales.to_excel(writer, sheet_name="City Sales", index=False)
    category_sales.to_excel(writer, sheet_name="Category Sales", index=False)
    delivery_report.to_excel(writer, sheet_name="Delivery Report", index=False)
    payment_report.to_excel(writer, sheet_name="Payment Report", index=False)
    monthly_sales.to_excel(writer, sheet_name="Monthly Trend", index=False)
    top_customers.to_excel(writer, sheet_name="Top Customers", index=False)

print("Excel report saved successfully")

Excel report saved successfully
